##### Camerano Alessio - Matricola 1044444
##### Corso di Elaborazione di Immagini e Visione Artificiale
---
# Riconoscimento di Sagome Arbitrarie tramite R-Table
---

### Cella 1: Caricamento Dataset ed Estrazione R-Tables Multi-Template
In questa cella viene inizializzato il flusso di elaborazione caricando e binarizzando l'intero dataset di test tramite la soglia automatica di Otsu. Successivamente, viene implementata la strategia multi-template per gestire la variabilità morfologica degli oggetti (es. ossa integre, spezzate o deformate). Per ciascun template presente, viene calcolato il centroide reale tramite i momenti geometrici di primo ordine, eliminando ogni offset globale arbitrario. Per attenuare l'aliasing, l'operatore di Sobel viene applicato sull'immagine binaria preventivamente sfocata con un filtro Gaussiano. Gli orientamenti del gradiente (\(\phi \)) vengono mappati in gradi interi nell'intervallo \([0, 360)\) per indicizzare i vettori geometrici \(r = y - x\) (dal pixel di bordo al centroide), conformemente alla Sezione 4.3 del paper di Ballard. Ogni modello viene memorizzato in un archivio strutturato contenente la propria R-table (R_table_np) e il numero esatto di campioni di contorno (vettori_totali_conteggio), garantendo un'allocazione coerente.

In [38]:
import time
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# =========================================================================
# FASE 1: CARICAMENTO DEL DATASET DI SCENA (EX CELLA INIZIALE)
# =========================================================================
path_scene = Path('train')
file_scene = list(path_scene.glob('*.[gG][iI][fF]'))
print(f"--- FASE 1: CARICAMENTO IMMAGINI DI TEST ---")
print(f"Trovate {len(file_scene)} immagini .gif nella cartella '{path_scene}'\n")

immagini_originali = {}
immagini_binarizzate = {}

for filepath in file_scene:
    nome_file = filepath.name 
    try:
        with Image.open(filepath) as img_gif:
            img_gray = np.array(img_gif.convert('L'))
    except Exception as e:
        print(f"Errore nel caricamento di {nome_file}: {e}")
        continue
        
    _, img_thresh = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    immagini_originali[nome_file] = img_gray
    immagini_binarizzate[nome_file] = img_thresh
    print(f"Caricata e binarizzata con successo: {nome_file}")

print(f"\nCaricamento dataset completato. Totale immagini pronte per il test: {len(immagini_binarizzate)}")
print("=" * 60 + "\n")


# =========================================================================
# FASE 2: ADDESTRAMENTO COMPOSITO MULTI-TEMPLATE (R-TABLES)
# =========================================================================
path_template = Path('template')
files_template = list(path_template.glob('*.[gG][iI][fF]'))

if len(files_template) == 0:
    raise ValueError("La cartella 'template' è vuota! Inserisci i file delle forme note (osso, cosciotta, ecc.).")

# Dizionario che conterrà le R-table di ogni singolo modello trovato
r_tables_archivio = {}

# Soglia unificata per garantire consistenza matematica tra training e test
SOGLIA_BORDO_UNIFICATA = 100

print("--- FASE 2: AVVIO ADDESTRAMENTO COMPOSITO SECONDO BALLARD ---")
print(f"Trovati {len(files_template)} modelli di template da analizzare.\n")

for filepath_template in files_template:
    nome_tpl = filepath_template.name
    try:
        with Image.open(filepath_template) as img_gif:
            img_gray = np.array(img_gif.convert('L'))
    except Exception as e:
        print(f"Errore nel caricamento del template {nome_tpl}: {e}")
        continue

    _, img_bin = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # Calcoliamo il centroide locale di questo specifico modello (Eq. 20)
    momenti = cv2.moments(img_bin)
    if momenti["m00"] == 0:
        print(f"Attenzione: il template {nome_tpl} non contiene pixel validi. Salto.")
        continue
    
    cx_ref = int(momenti["m10"] / momenti["m00"])
    cy_ref = int(momenti["m01"] / momenti["m00"])

    # Estrazione dei gradienti locali sul modello
    img_smooth = cv2.GaussianBlur(img_bin, (3, 3), 0)
    gx = cv2.Sobel(img_smooth, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(img_smooth, cv2.CV_64F, 0, 1, ksize=3)
    magnitudo = np.sqrt(gx**2 + gy**2)
    orientamento_deg = np.degrees(np.arctan2(gy, gx)) % 360

    pixel_bordo = magnitudo > SOGLIA_BORDO_UNIFICATA
    righe_bordo, colonne_bordo = np.where(pixel_bordo)

    vettori_singoli = []
    for r, c in zip(righe_bordo, colonne_bordo):
        phi = int(round(orientamento_deg[r, c])) % 360
        
        # Sez. 4.3: r = y - x (vettore dal pixel di bordo x al centroide locale y)
        dr = cy_ref - r
        dc = cx_ref - c
        
        # Salviamo la riga della R-table: (phi, dr, dc, peso_formale=1.0)
        vettori_singoli.append([phi, dr, dc, 1.0])

    R_table_np = np.array(vettori_singoli, dtype=np.float32)
    
    # Archiviamo l'R-table con le sue informazioni di normalizzazione specifiche
    r_tables_archivio[nome_tpl] = {
        "table": R_table_np,
        "conteggio": R_table_np.shape[0]  # Salviamo l'intero esatto (numero righe)
    }
    print(f"Template '{nome_tpl}' integrato con successo. Vettori totali: {R_table_np.shape[0]}")

print("\n==================================================")
print(f" TRAINING COMPLETATO. Archivio modelli configurato.")
print(f" Modelli totali registrati nell'archivio: {len(r_tables_archivio)}")
print("==================================================")


--- FASE 1: CARICAMENTO IMMAGINI DI TEST ---
Trovate 180 immagini .gif nella cartella 'train'

Caricata e binarizzata con successo: Bone-3.gif
Caricata e binarizzata con successo: apple-1.gif
Caricata e binarizzata con successo: apple-2.gif
Caricata e binarizzata con successo: apple-3.gif
Caricata e binarizzata con successo: apple-4.gif
Caricata e binarizzata con successo: apple-5.gif
Caricata e binarizzata con successo: apple-6.gif
Caricata e binarizzata con successo: apple-7.gif
Caricata e binarizzata con successo: apple-8.gif
Caricata e binarizzata con successo: apple-9.gif
Caricata e binarizzata con successo: apple-10.gif
Caricata e binarizzata con successo: apple-11.gif
Caricata e binarizzata con successo: apple-12.gif
Caricata e binarizzata con successo: apple-13.gif
Caricata e binarizzata con successo: apple-14.gif
Caricata e binarizzata con successo: apple-15.gif
Caricata e binarizzata con successo: apple-16.gif
Caricata e binarizzata con successo: apple-17.gif
Caricata e binar

### Cella 2: Accumulatore 4D, Votazione Geometrica e Normalizzazione Formale
Il processo di localizzazione esplora lo spazio dei parametri quadridimensionale \(A(y, s, \theta)\) campionando esplicitamente le variazioni di scala geometrica (\(s\)) e rotazione (\(\theta \)), oltre a gestire l'inversione figura-sfondo mediante la traslazione degli indici di orientamento di \(180^{\circ }\) (Sezione 4.5). L'operazione di voto proietta i vettori della R-table nella scena calcolando le coordinate Hough tramite broadcasting cartesiano bidimensionale. Al fine di risolvere i problemi di allineamento e ottimizzare le prestazioni, l'array dei pesi formali \(w_{i}\) viene espanso dinamicamente sulla forma della matrice dei centri validi tramite np.broadcast_to, scongiurando errori di indicizzazione booleana. Per contrastare la dispersione dei voti causata dal rumore e dalle imperfezioni dei contorni, l'accumulatore subisce uno smoothing spaziale con kernel Gaussiano esteso a (11, 11), fungendo da integratore d'area (Sezione 2.2, Eq. 5). La metrica di confidenza finale viene calcolata normalizzando il picco massimo assoluto dei voti sul numero totale di vettori effettivi del rispettivo template, consentendo l'adozione di una soglia empirica al 5% altamente sensibile e robusta contro i falsi positivi generati dallo sfondo.

In [39]:
import time

soglia_confidenza = 0.05  # Con i template corretti, possiamo rimettere una soglia solida!
totale_immagini = len(immagini_binarizzate)
immagini_con_oggetto = []

scale_test    = [0.8, 1.0, 1.2]
rotazioni_test = list(range(0, 360, 30))
stati_inversione_fg = [False, True]

num_scale     = len(scale_test)
num_rotazioni = len(rotazioni_test)

t_inizio_totale = time.time()
print("--- AVVIO ACCUMULATORE FORMALE MULTI-TEMPLATE ---")

for nome_scena, img_scena_bin in immagini_binarizzate.items():
    t_scena = time.time()
    altezza, larghezza = img_scena_bin.shape

    # Estrazione bordi scena
    img_scena_smooth = cv2.GaussianBlur(img_scena_bin, (3, 3), 0)
    gx_scena = cv2.Sobel(img_scena_smooth, cv2.CV_64F, 1, 0, ksize=3)
    gy_scena = cv2.Sobel(img_scena_smooth, cv2.CV_64F, 0, 1, ksize=3)
    magnitudo_scena = np.sqrt(gx_scena**2 + gy_scena**2)
    orientamento_deg_scena = np.degrees(np.arctan2(gy_scena, gx_scena)) % 360

    pixel_bordo_scena = magnitudo_scena > SOGLIA_BORDO_UNIFICATA
    righe_scena, colonne_scena = np.where(pixel_bordo_scena)

    if len(righe_scena) == 0:
        continue

    orientamenti_bordo = orientamento_deg_scena[pixel_bordo_scena].astype(np.int32)
    
    mappa_pixel_scena = {}
    for phi in range(360):
        scelta = (orientamenti_bordo == phi)
        if np.any(scelta):
            mappa_pixel_scena[phi] = (righe_scena[scelta], colonne_scena[scelta])

    # Tracciamento del miglior risultato assoluto per QUESTA scena tra tutti i template
    max_confidenza_scena = 0.0
    miglior_template_match = None
    miglior_scala = None
    miglior_rotazione = None

    # CICLO CRUCIALE: testiamo la scena contro ogni forma nota nel nostro archivio
    for nome_tpl, tpl_data in r_tables_archivio.items():
        R_table_np = tpl_data["table"]
        vettori_totali_conteggio = tpl_data["conteggio"]

        phi_template = R_table_np[:, 0].astype(np.int32)
        dr_template  = R_table_np[:, 1]
        dc_template  = R_table_np[:, 2]
        w_template   = R_table_np[:, 3]

        accumulatore_4d = np.zeros((altezza, larghezza, num_scale, num_rotazioni), dtype=np.float32)

        for invertire_fg in stati_inversione_fg:
            phi_lavorazione = (phi_template + 180) % 360 if invertire_fg else phi_template.copy()

            mappa_vettori_template = {}
            for phi in range(360):
                scelta_v = (phi_lavorazione == phi)
                if np.any(scelta_v):
                    mappa_vettori_template[phi] = (dr_template[scelta_v], dc_template[scelta_v], w_template[scelta_v])

            for idx_s, s in enumerate(scale_test):
                for idx_theta, theta in enumerate(rotazioni_test):
                    theta_rad = np.radians(theta)
                    cos_t, sin_t = np.cos(theta_rad), np.sin(theta_rad)

                    acc_temporaneo_voto = np.zeros((altezza, larghezza), dtype=np.float32)

                    for phi_target in mappa_pixel_scena.keys():
                        r_scena_sub, c_scena_sub = mappa_pixel_scena[phi_target]
                        phi_mod = (phi_target - theta) % 360
                        if phi_mod not in mappa_vettori_template:
                            continue

                        dr_sub, dc_sub, w_sub = mappa_vettori_template[phi_mod]
                        dr_trasformati = s * (dr_sub * cos_t - dc_sub * sin_t)
                        dc_trasformati = s * (dr_sub * sin_t + dc_sub * cos_t)

                        centri_r = r_scena_sub[:, None] + dr_trasformati[None, :]
                        centri_c = c_scena_sub[:, None] + dc_trasformati[None, :]
                        pesi_voto = np.broadcast_to(w_sub[None, :], centri_r.shape)

                        centri_r_int = np.round(centri_r).astype(np.int32)
                        centri_c_int = np.round(centri_c).astype(np.int32)

                        validi = (centri_r_int >= 0) & (centri_r_int < altezza) & (centri_c_int >= 0) & (centri_c_int < larghezza)
                        if np.any(validi):
                            np.add.at(acc_temporaneo_voto, (centri_r_int[validi], centri_c_int[validi]), pesi_voto[validi])

                    accumulatore_4d[:, :, idx_s, idx_theta] = np.maximum(accumulatore_4d[:, :, idx_s, idx_theta], acc_temporaneo_voto)

        # Verifica dei picchi per questo specifico template
        for idx_s in range(num_scale):
            for idx_theta in range(num_rotazioni):
                acc_piano = accumulatore_4d[:, :, idx_s, idx_theta]
                acc_piano_smooth = cv2.GaussianBlur(acc_piano, (7, 7), 0)
                _, max_voti, _, _ = cv2.minMaxLoc(acc_piano_smooth)

                punteggio_corrente = max_voti / vettori_totali_conteggio if vettori_totali_conteggio > 0 else 0.0

                if punteggio_corrente > max_confidenza_scena:
                    max_confidenza_scena = punteggio_corrente
                    miglior_template_match = nome_tpl
                    miglior_scala = scale_test[idx_s]
                    miglior_rotazione = rotazioni_test[idx_theta]

    t_fine_scena = time.time() - t_scena

    if max_confidenza_scena >= soglia_confidenza:
        immagini_con_oggetto.append(nome_scena)
        print(f"[TROVATO] '{nome_scena}' - Confidenza: {max_confidenza_scena:.2%}")
        print(f"          > Template: {miglior_template_match} | Scala: {miglior_scala} | Rotazione: {miglior_rotazione}° | Tempo: {t_fine_scena:.2f}s\n")
    else:
        print(f"[NON TROVATO] '{nome_scena}' - Escluso (Confidenza max: {max_confidenza_scena:.2%}) | Tempo: {t_fine_scena:.2f}s\n")

print("\n==================================================")
print(f" REPORT FINALE - Tempo Totale: {time.time() - t_inizio_totale:.2f}s")
print(f" Immagini positive identificate: {len(immagini_con_oggetto)}")
print("==================================================")


--- AVVIO ACCUMULATORE FORMALE MULTI-TEMPLATE ---
[NON TROVATO] 'Bone-3.gif' - Escluso (Confidenza max: 2.25%) | Tempo: 1.40s

[NON TROVATO] 'apple-1.gif' - Escluso (Confidenza max: 1.13%) | Tempo: 0.64s

[NON TROVATO] 'apple-2.gif' - Escluso (Confidenza max: 1.05%) | Tempo: 0.62s

[NON TROVATO] 'apple-3.gif' - Escluso (Confidenza max: 1.10%) | Tempo: 0.67s

[NON TROVATO] 'apple-4.gif' - Escluso (Confidenza max: 1.15%) | Tempo: 0.64s

[NON TROVATO] 'apple-5.gif' - Escluso (Confidenza max: 1.22%) | Tempo: 0.75s

[NON TROVATO] 'apple-6.gif' - Escluso (Confidenza max: 1.49%) | Tempo: 0.63s

[NON TROVATO] 'apple-7.gif' - Escluso (Confidenza max: 1.15%) | Tempo: 0.71s

[NON TROVATO] 'apple-8.gif' - Escluso (Confidenza max: 1.44%) | Tempo: 0.63s

[NON TROVATO] 'apple-9.gif' - Escluso (Confidenza max: 1.44%) | Tempo: 0.67s

[NON TROVATO] 'apple-10.gif' - Escluso (Confidenza max: 1.14%) | Tempo: 0.67s

[NON TROVATO] 'apple-11.gif' - Escluso (Confidenza max: 0.99%) | Tempo: 0.69s

[NON TROVATO]

### SEZIONE CRITICITÀ E LIMITAZIONI

* **Complessità computazionale dello spazio dei parametri:** Sebbene la R-Table consenta di mappare variazioni geometriche tramite trasformazioni lineari (Sez. 4.5), l'estensione dell'algoritmo a forme scalate e ruotate richiede un accumulatore quadridimensionale $A(y, s, \theta)$. Ballard stesso rileva come l'introduzione di ciascun parametro libero causi un incremento esponenziale del costo computazionale (proporzionale a $M^{m-2}$). Nel codice, l'uso del broadcasting NumPy ottimizza la memoria, ma il calcolo nidificato su scale, rotazioni ed inversioni figura-sfondo satura rapidamente i tempi di calcolo della CPU in presenza di dataset estesi.

* **Arbitrarietà e dipendenza dalle soglie di magnitudo:** L'algoritmo mostra un'elevata sensibilità alla scelta delle soglie operative sul gradiente. L'adozione di una soglia unificata (es. `SOGLIA_BORDO_UNIFICATA = 100`) è indispensabile per preservare la consistenza matematica tra i vettori memorizzati e i pixel votanti della scena. Tuttavia, in presenza di oggetti parzialmente occlusi o con contorni sfumati (come le variazioni morfologiche delle ossa), una soglia fissa causa la perdita di vettori critici, frammentando la convergenza dei voti nel piano Hough e forzando l'abbassamento empirico della soglia di confidenza al 5%.

* **Confronto con architetture di Deep Learning:** Rispetto ai moderni modelli di Object Detection e Instance Segmentation basati su Convolutional Neural Networks (CNN) o Vision Transformers (ViT), l'approccio geometrico classico di Ballard mostra limiti evidenti nell'estrazione semantica delle feature. Architetture basate su **Focal Loss** superano nativamente il problema dello sbilanciamento tra primo piano e sfondo, concentrando i gradienti di errore sugli esempi negativi difficili (*hard negatives*) durante il training. Al contrario, la natura puramente additiva della Generalized Hough Transform (Eq. 24), pur supportando pesi formali $w_i$ (Sez. 5.2), non possiede un meccanismo di penalizzazione dinamica del rumore, rendendo il sistema vulnerabile ad accumuli casuali di voto generati da texture complesse o artefatti di binarizzazione nella scena.
